# AVENIQ SENTINEL // PHASE 3.1: ML FEATURE CONTRACT & LEAKAGE VALIDATION
## Formal Feature Schema, Type Enforcement & Leakage Shield

**Product:** Aveniq Sentinel by Aveniq  
**Target Repository:** https://github.com/sparxweb/aveniq-sentinel  

---

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path('.').resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
from neuralshield.ml.feature_contract import FeatureContract
from neuralshield.ml.experiment import ExperimentContract, ExperimentMetadata
from config.settings import SAMPLE_DATA_DIR

print('=== ML FEATURE CONTRACT ENGINE LOADED ===')

=== ML FEATURE CONTRACT ENGINE LOADED ===


In [ ]:
contract = FeatureContract()
df = pd.read_parquet(SAMPLE_DATA_DIR / 'cic_ids2017_sample.parquet')

audit = contract.validate_dataframe(df)
print(f'Contract Features Defined: {audit["total_contract_features"]}')
print(f'Available Permitted Features: {audit["available_permitted_count"]}')
print(f'Detected Leakage Columns: {audit["detected_leakage_columns"]}')
print(f'Leakage Hazard Present: {audit["has_leakage_hazard"]}')
print(f'Approved for Training: {audit["is_valid_for_training"]}')

Contract Features Defined: 29
Available Permitted Features: 20
Detected Leakage Columns: []
Leakage Hazard Present: False
Approved for Training: True


In [ ]:
# Extract sanitized feature matrix X and target vector y
X, y = contract.prepare_feature_matrix(df, label_col='Label')
print(f'Feature Matrix X Shape: {X.shape} (All float32: {all(X.dtypes == np.float32)})')
print(f'Target Vector y Shape:   {y.shape}')
print(f'Target Class Counts:\n{y.value_counts()}')

Feature Matrix X Shape: (1505, 37) (All float32: True)
Target Vector y Shape:   (1505,)
Target Class Counts:
Label
BENIGN      1193
DDoS         158
PortScan     112
Bot           42
Name: count, dtype: int64


In [ ]:
# Create and verify reproducibility experiment metadata manifest
meta = ExperimentMetadata(
    experiment_id='PHASE_3_1_VERIFICATION_RUN',
    created_at='2026-11-23T18:00:00Z',
    dataset_name='CIC-IDS2017-Sample',
    dataset_version='1.0-deterministic',
    feature_schema_version='3.1.0',
    train_period=['Monday', 'Tuesday', 'Wednesday'],
    val_period=['Wednesday-Afternoon'],
    test_period=['Thursday', 'Friday'],
    model_type='Baseline-Scaffolding',
    hyperparameters={'chunk_size': 100_000, 'random_state': 42},
    metrics={'status': 'schema_verified'},
    target_classes=sorted(list(y.unique())),
    unseen_test_classes=['Web Attack', 'Botnet', 'PortScan', 'DDoS'],
    random_seed=42,
    python_version=sys.version.split()[0],
    sklearn_version='1.5.2',
    environment='Local-Audit',
)

out_dir = PROJECT_ROOT / 'artifacts' / 'manifests'
manifest_path = ExperimentContract.save_experiment_manifest(meta, out_dir)
print(f'Saved verified manifest: {manifest_path}')

Saved verified manifest: C:\Users\HP\OneDrive\Desktop\NEURALSHIELD AI\artifacts\manifests\PHASE_3_1_VERIFICATION_RUN_manifest.json
